# PEER+TEXT: gold-free NL→FOL faithfulness checks on a held-out test set (demo)

This notebook is a runnable walk-through of `method.py`, the driver script of the experiment **"Peer agreement plus text checks, held-out test"**.

**The problem.** Given a natural-language sentence and a candidate first-order-logic (FOL) formula from *any* system, predict whether the formula is **faithful** to the sentence. There is no gold formula, no ontology and no domain knowledge.

**The method (PEER+TEXT)** combines two groups of signals:
* **PEER**: graded agreement across model families. Formulas from *other* model families (the peers) are aligned into the candidate's vocabulary. The candidate is split into claim units, and a finite-model refuter plus z3 checks whether each unit is entailed. This gives `support` / `coverage` / `g_score` and per-unit error codes. `c_score_align` is the binary agreement score from iteration 1.
* **TEXT**: solver-exact checks from iteration 1 that compare the formula with the text alone. `l2_bow` checks that the text's content is accounted for. `l3_z3` compares a text-only questionnaire with the z3 role profile of the formula.
* A **logistic fusion** of `g_score`, `c_score_align` and `l2_bow`. Its parameters were frozen on the iteration-1 screen and pre-registered (sha256 `b9f28b1b…`), and it was scored **once** on the held-out dataset **E** (8,507 candidate rows, 700 sentences).

**Headline results (full run, R_AB = label tiers A+B, 1822 ERROR / 864 CORRECT):** PEER+TEXT AUROC **0.790** [0.75, 0.82]. For comparison: c_score_align 0.782, TEXT 0.693, and a local Qwen3-8B disguised judge 0.712 (delta +0.078 [0.043, 0.112]).

**What runs here.** The original `method.py` runs these stages as subprocesses: data views, OpenRouter LLM calls, a GPU judge, the screen fit, solver scoring and the analysis. These stages need API keys, a GPU and hours of compute, so the notebook cannot run them. It runs the one stage `method.py` implements itself, **`build_outputs`**. That stage joins the precomputed per-item scores with the labels **after** the analysis. The notebook runs it on a curated subset of 100 rows of E (`mini_demo_data.json`). It then re-derives the frozen fusion score from its parameters and compares every metric's AUROC on the demo subset with the full-run headline numbers.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT on Colab, always install
_pip('loguru==0.7.3')

# numpy, pandas, sklearn, matplotlib — pre-installed on Colab, install locally only
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'matplotlib==3.10.0')

## Imports
The first block is the original import block of `method.py`, unchanged. The second block adds what the notebook needs for the fusion re-derivation and the plots (`math`, `numpy`, `pandas`, `sklearn`, `matplotlib`).

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

from loguru import logger

# --- additional imports for the notebook (fusion re-derivation + visualisation) ---
import math
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

## Data loading
`mini_demo_data.json` holds a curated subset of dataset E: 100 rows, with ERROR and CORRECT balanced in each of the strata L25, L20, EXC and CTRL, plus 6 unparseable rows and 6 tier-C or contested rows. It contains:
* `per_item_E`: the per-row score records from `results/per_item_E.jsonl` (all metrics, judges, unit codes, strata)
* `E_blind`: the blind view (text, candidate FOL, system) from `data/E_blind.jsonl`
* `E_labels`: the labels (final label, tier, error ops) from `data/E_labels.jsonl`
* `analysis`: `a_tables` and `criteria` from `results/analysis.json`, which are the full-run results
* `prereg`: the frozen fusion and text-only models and the imputation rules from `results/prereg.json`
* `p_tests`: the verdicts on the predictions (`results/p_tests.json`), and `prereg_sha256`

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-eff060-layered-gold-free-checks-for-logic/fork/run_StxQVNnY7aQW/round-2/experiment-5/demo/mini_demo_data.json"
import json
from pathlib import Path

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    local = Path("mini_demo_data.json")
    if local.exists(): return json.loads(local.read_text())
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print({k: (len(v) if isinstance(v, (list, dict)) else v) for k, v in data.items() if k != "description"})

## Configuration
All tunable parameters are set here.
* `STAGE`: the pipeline stage passed to `main()`. It replaces `sys.argv[1]` from the original script. The original default is `"all"`, which runs every subprocess stage and needs `src/*.py`, an OpenRouter key and a GPU. Only `"outputs"` can run in this notebook.
* `N_ROWS`: how many of the 100 demo rows to process. The original processes all 8,507 rows of E.
* `N_BOOT`: the number of bootstrap resamples for the demo AUROC confidence intervals in the final cell. The original analysis (`src/analyse_E.py`) uses 2000 for the headline tables.

In [ ]:
STAGE = "outputs"   # original default: "all" (needs src/*.py, API key, GPU — not available here)
N_ROWS = 20         # rows of the demo subset to process (original: all 8,507 rows of E)
N_BOOT = 20         # bootstrap resamples for the demo AUROC CIs (original analysis: 2000)

## Setup: paths and logging
This is the original module-level setup. `ROOT` was the script's directory. Here it is the notebook's working directory, and `logs/` is created so the rotating log file can be written. `PY` and `PY_GPU` point at the two virtual environments of the original project. They are only used by the subprocess stages, which this notebook does not run.

In [ ]:
ROOT = Path.cwd()  # original: Path(__file__).resolve().parent
(ROOT / "logs").mkdir(exist_ok=True)  # notebook fix: the log directory must exist
PY = str(ROOT / ".venv" / "bin" / "python")
PY_GPU = str(ROOT / ".venv_gpu" / "bin" / "python")

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")
logger.add(ROOT / "logs" / "method.log", rotation="30 MB", level="DEBUG")

## Helpers
These are the original helpers, unchanged:
* `sh`: runs one pipeline stage as a subprocess and fails loudly on a nonzero exit.
* `jl`: reads a JSONL file. The notebook reads from the in-memory `data` instead of files.
* `s`: formats a score as a string, with `"NA"` for a missing value and 6 decimals for floats. This is the `predict_*` format of the `exp_gen_sol_out` schema.

In [ ]:
def sh(args: list[str]) -> None:
    logger.info("$ " + " ".join(args))
    r = subprocess.run(args, cwd=ROOT)
    if r.returncode != 0:
        raise RuntimeError(f"stage failed ({r.returncode}): {' '.join(args)}")


def jl(p: Path) -> list[dict]:
    return [json.loads(l) for l in p.read_text().splitlines() if l.strip()] if p.exists() else []


def s(x) -> str:
    if x is None:
        return "NA"
    if isinstance(x, float):
        return f"{x:.6f}"
    return str(x)

## `build_outputs`: join scores with labels after the analysis
This is the core of `method.py`. For every row of E it builds one example:
* `input`: the blind view of the row (text, candidate FOL, system, prompt variant, keys) as JSON
* `output`: the final label (`ERROR` / `CORRECT` / `UNPARSEABLE` / `CONTESTED` / …), joined only after the analysis
* `predict_*`: every metric's score, oriented so that **higher = more likely an error**. The metrics are PEER+TEXT (`p_peer_text`), PEER only, TEXT only, `g_score`, the name-free and aligned consensus scores, L2-bow, L3, L1 lint, L2-role, and the four LLM judges (cheap flash-lite and local Qwen3-8B, each with disguised and original prompts).
* `metadata_*`: strata, fold, label tier, unit error codes, fired signal, cost and time

The code is the original. The only changes are that the five file reads now take their values from the loaded `data` dict, and that the rows are capped at `N_ROWS`.

In [ ]:
def build_outputs() -> None:
    """method_out.json in the exp_gen_sol_out schema: one example per E row (input = blind-view JSON, output = the E
    final label joined AFTER the analysis), predict_* = every metric's score (higher = more likely an error)."""
    items = data["per_item_E"][:N_ROWS]  # original: jl(ROOT / "results" / "per_item_E.jsonl")
    labels = {r["row_key"]: r for r in data["E_labels"]}  # original: jl(ROOT / "data" / "E_labels.jsonl")
    blind = {r["row_key"]: r for r in data["E_blind"]}  # original: jl(ROOT / "data" / "E_blind.jsonl")
    analysis = data["analysis"]  # original: json.loads((ROOT / "results" / "analysis.json").read_text())
    pre = data["prereg"]  # original: json.loads((ROOT / "results" / "prereg.json").read_text())
    ex = []
    for r in items:
        b = blind[r["row_key"]]
        L = labels[r["row_key"]]
        inp = {"text": b["text"], "candidate_fol": b["candidate_fol"], "system": b["system"], "prompt_variant": b["prompt_variant"],
               "row_key": r["row_key"], "item_id": b["item_id"], "sentence_id": b["sentence_id"]}
        e = {"input": json.dumps(inp, ensure_ascii=False), "output": L["final_label"],
             "predict_peer_text": s(r.get("p_peer_text")), "predict_peer_text_flag": s(r.get("flag")),
             "predict_peer": s(r.get("peer_only")), "predict_text": s(r.get("p_text")),
             "predict_g_score": s(r.get("g_score")), "predict_c_score_nf": s(r.get("c_score_nf")),
             "predict_c_score_align": s(r.get("c_score_align")), "predict_nf_g_score": s(r.get("nf_g_score")),
             "predict_nf_c_score": s(r.get("nf_c_score")), "predict_l2_bow": s(r.get("l2_bow")), "predict_l3": s(r.get("l3_z3")),
             "predict_l1_any": s(r.get("l1_any")), "predict_l2_role": s(r.get("l2_role")),
             "predict_judge_cheap_disg": s(r.get("judge_cheap_disg")), "predict_judge_cheap_orig": s(r.get("judge_cheap_orig")),
             "predict_judge_local_qwen8b_disg": s(r.get("judge_local_qwen8b_disg")),
             "predict_judge_local_qwen8b_orig": s(r.get("judge_local_qwen8b_orig")),
             "metadata_row_key": r["row_key"], "metadata_item_id": r["item_id"], "metadata_sentence_id": r["sentence_id"],
             "metadata_system": r["system"], "metadata_system_class": r["system_class"], "metadata_family_vendor": r["family_vendor"],
             "metadata_stratum": r["stratum"], "metadata_strata": r["strata"], "metadata_fold_E": r["fold_E"],
             "metadata_label_tier": L["label_tier"], "metadata_repair_ops": L["repair_ops"], "metadata_error_ops": L["error_ops"],
             "metadata_reading_choice": L["reading_choice"], "metadata_coverage_status": r["coverage_status"],
             "metadata_unit_codes": [c["code"] + ":" + c.get("unit", "")[:120] for c in (r.get("unit_codes") or [])][:12],
             "metadata_top_code": r.get("top_code"), "metadata_fired_signal": r.get("fired_signal"),
             "metadata_model_used": r.get("model_used"), "metadata_n_unknown": r.get("n_unknown"),
             "metadata_n_peers_used": r.get("n_peers_used"), "metadata_peer_unavailable": r.get("peer_unavailable"),
             "metadata_support": r.get("support"), "metadata_coverage": r.get("coverage"),
             "metadata_medoid_ops": r.get("medoid_ops"), "metadata_judge_local_type": r.get("judge_local_qwen8b_disg_type"),
             "metadata_cost_usd": (r.get("cost_usd_peer_text") or 0.0), "metadata_judge_cost_usd": r.get("judge_cost_usd"),
             "metadata_secs": (r.get("secs_pairs") or 0.0) + (r.get("secs_text") or 0.0),
             "metadata_prereg_sha256": r["prereg_sha256"]}
        ex.append(e)
    heads = {k: {kk: vv for kk, vv in v.items() if kk in ("n", "n_error", "n_correct", "metrics")} for k, v in analysis["a_tables"].items()}
    out = {"metadata": {
        "method_name": "PEER+TEXT (graded cross-family consensus fused with solver-exact text checks), frozen on the screen, scored once on E",
        "prereg_sha256": data["prereg_sha256"].split()[0],  # original: (ROOT / "results" / "prereg.sha256").read_text().split()[0]
        "peer_variant": pre["variant"], "F1_failed_on_screen": pre["selection_evidence"]["F1_failed"],
        "fusion": pre["frozen"]["fusion"], "text_only": pre["frozen"]["text_only"],
        "orientation": "every predict_* is oriented so that higher = more likely an error; NA = not available",
        "headline_auroc_tables": heads, "criteria": analysis["criteria"],
        "p_tests": data["p_tests"],  # original: json.loads((ROOT / "results" / "p_tests.json").read_text())
        "workspace": str(ROOT)},
        "datasets": [{"dataset": "E_heldout_candidates", "examples": ex}]}
    (ROOT / "method_out.json").write_text(json.dumps(out, ensure_ascii=False))
    logger.info(f"method_out.json: {len(ex)} examples, {(ROOT / 'method_out.json').stat().st_size / 1e6:.1f} MB")

## `main`: the stage dispatcher
This is the original dispatcher. Every stage except `outputs` starts one of the project's `src/*.py` scripts as a subprocess:

| stage | script | what it does | needs |
|---|---|---|---|
| `views` | `src/data_views.py` | blind view and label view of E, API row order | data |
| `api` | `src/run_api_E.py` | L3 text questionnaires and flash-lite judges | OpenRouter key |
| `local` | `src/local_judge_E.py` | local Qwen3-8B judge | GPU |
| `screen` | `src/screen_fit.py` | screen features, variant selection, fusion fit, pre-registration freeze | screen data |
| `score` | `src/score_E.py` | PEER + TEXT solver scoring of E (mini → 100 → all), assembly | peers, z3 |
| `analyse` | `src/analyse_E.py` | the only label join: AUROCs, CIs, P1–P4 tests | labels + prereg hash |
| `outputs` | (in this file) | `build_outputs()` | results |

The only change is that the stage comes from the config cell instead of `sys.argv`. With `STAGE = "outputs"` only `build_outputs()` runs.

In [ ]:
@logger.catch(reraise=True)
def main():
    stage = STAGE  # original: sys.argv[1] if len(sys.argv) > 1 else "all"
    if stage in ("views", "all"):
        sh([PY, "src/data_views.py"])
    if stage in ("api", "all"):
        sh([PY, "src/run_api_E.py"])
    if stage in ("local", "all") and Path(PY_GPU).exists():
        sh([PY_GPU, "src/local_judge_E.py"])
    if stage in ("screen", "all"):
        if not (ROOT / "results" / "prereg.json").exists():
            sh([PY, "src/screen_fit.py", "compute"])
            sh([PY, "src/screen_fit.py", "fit"])
            sh([PY, "src/screen_fit.py", "freeze"])
    if stage in ("score", "all"):
        for st in ("mini", "100", "all"):
            sh([PY, "src/score_E.py", "run", "--stage", st])
        sh([PY, "src/score_E.py", "assemble"])
    if stage in ("analyse", "all"):
        sh([PY, "src/analyse_E.py"])
    if stage in ("outputs", "all"):
        build_outputs()


main()
out = json.loads((ROOT / "method_out.json").read_text())
ex = out["datasets"][0]["examples"]
print(json.dumps({k: v for k, v in ex[0].items() if k.startswith(("input", "output", "predict_peer", "predict_text", "predict_c_score_align"))}, indent=1, ensure_ascii=False)[:1500])

## Re-deriving the frozen PEER+TEXT fusion score
`p_peer_text` in the per-item records was computed by `fused_score` in `src/peer_text.py`. The three functions below are copied verbatim from that file. The fusion is a logistic regression on the standardized features `ALIGN:g_score` (the per-item `g_score`), `c_score_align` and `l2_bow`. A missing feature gets z = 0. If the peer feature is missing, the frozen TEXT-only model (`l2_bow`, `l3_z3`) is used instead, and an unparseable formula gets p = 1. Applying the frozen parameters from the pre-registration to the stored features should reproduce the stored `p_peer_text` exactly.

In [ ]:
# ---- copied verbatim from src/peer_text.py ----
def _z(x, mu, sd):
    return 0.0 if x is None else (x - mu) / (sd if sd > 0 else 1.0)


def apply_logistic(model: dict, feats: dict) -> float:
    """model = {'features', 'mean', 'sd', 'coef', 'intercept'}; missing feature -> z = 0."""
    s = model["intercept"]
    for f, m, sd, c in zip(model["features"], model["mean"], model["sd"], model["coef"]):
        s += c * _z(feats.get(f), m, sd)
    return 1 / (1 + math.exp(-s))


def fused_score(frozen: dict, feats: dict) -> dict:
    """Frozen PEER+TEXT fusion. Unparseable -> p_error 1.0. peer_unavailable -> frozen TEXT-only model (pre-registered).
    fired_signal = argmax standardized contribution (PEER = g/c_score_nf features, TEXT = l2_bow/l3)."""
    if feats.get("coverage_status") == "UNPARSEABLE":
        return {"p_error": 1.0, "flag": 1, "fired_signal": "UNPARSEABLE", "model_used": "none"}
    main = frozen["fusion"]
    use = main
    if feats.get("peer_unavailable") or feats.get(main["features"][0]) is None:
        use = frozen["text_only"]
    p = apply_logistic(use, feats)
    contrib = {f: c * _z(feats.get(f), m, sd) for f, m, sd, c in zip(use["features"], use["mean"], use["sd"], use["coef"])}
    top = max(contrib, key=contrib.get) if contrib else None
    sig = "PEER" if top in ("g_score", "c_score_nf", "c_score_align") else ("TEXT" if top else None)
    thr = main["threshold"] if use is main else frozen["text_only"]["threshold"]
    return {"p_error": p, "flag": int(p >= thr), "fired_signal": sig, "model_used": "fusion" if use is main else "text_only"}
# -----------------------------------------------

frozen = data["prereg"]["frozen"]
print("fusion features:", frozen["fusion"]["features"], "coef:", np.round(frozen["fusion"]["coef"], 3), "threshold:", round(frozen["fusion"]["threshold"], 3))
max_diff, n_used = 0.0, {}
for r in data["per_item_E"][:N_ROWS]:
    # per-item records store the ALIGN variant's g_score as `g_score`; the frozen model names it "ALIGN:g_score"
    feats = {"ALIGN:g_score": r["g_score"], "c_score_align": r["c_score_align"], "l2_bow": r["l2_bow"], "l3_z3": r["l3_z3"],
             "peer_unavailable": r["peer_unavailable"], "coverage_status": r["coverage_status"]}
    fs = fused_score(frozen, feats)
    max_diff = max(max_diff, abs(fs["p_error"] - r["p_peer_text"]))
    n_used[fs["model_used"]] = n_used.get(fs["model_used"], 0) + 1
print(f"re-derived vs stored p_peer_text: max |diff| = {max_diff:.2e} over {N_ROWS} rows; model used: {n_used}")

## Results: metric AUROCs on the demo subset vs. the full held-out run
The final cell applies the same evaluation regime as the paper (**R_AB**): LLM-generated candidates only, label tier A or B, `ERROR` (y = 1) vs `CORRECT` (y = 0), and reading-choice rows excluded. Missing scores are imputed with the pre-registered neutral values, following `src/analyse_E.py`. The cell then computes each metric's AUROC with a bootstrap CI on the demo rows and plots it next to the full-run AUROC from `analysis.json` (n = 2,686).

With about 80 evaluable demo rows the CIs are wide. **The full-run numbers are the reference.** The demo subset is also balanced by stratum and label, so its AUROCs will not match the full run exactly.

In [ ]:
df = pd.DataFrame(data["per_item_E"][:N_ROWS])
lab = pd.DataFrame(data["E_labels"][:N_ROWS])[["row_key", "final_label", "label_tier", "reading_choice"]]
df = df.merge(lab, on="row_key")
df["y"] = np.where(df.final_label == "ERROR", 1, np.where(df.final_label == "CORRECT", 0, -1))
rab = df[(df.system_class == "llm") & (~df.reading_choice.astype(bool)) & df.label_tier.isin(["A", "B"]) & (df.y >= 0)].copy()

neutral = data["prereg"]["imputation"]["neutral"]
IMPUTE = {"g_score": neutral["ALIGN:g_score"], "c_score_align": neutral["c_score_align"], "l2_bow": neutral["l2_bow"],
          "l3_z3": neutral["l3_z3"], "nf_c_score": neutral["NF-anchored:c_score_nf"], "peer_only": neutral["ALIGN:g_score"]}
METRICS = {"p_peer_text": "PEER+TEXT (ours)", "c_score_align": "c_score_align (iter-1)", "peer_only": "PEER only",
           "p_text": "TEXT only", "l2_bow": "L2-bow", "l3_z3": "L3", "nf_c_score": "NF-anchored c_score",
           "judge_local_qwen8b_disg": "Qwen3-8B judge (disg.)"}
for c, v in IMPUTE.items():
    rab[c] = rab[c].fillna(v)

rng = np.random.default_rng(0)
full = data["analysis"]["a_tables"]["R_AB pooled"]["metrics"]
rows = []
for m, name in METRICS.items():
    sub = rab[rab[m].notna()]
    y, x = sub.y.values, sub[m].values.astype(float)
    if len(set(y)) < 2:
        continue
    auc = roc_auc_score(y, x)
    boots = []
    for _ in range(N_BOOT):
        i = rng.integers(0, len(y), len(y))
        if len(set(y[i])) == 2:
            boots.append(roc_auc_score(y[i], x[i]))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    f = full.get(m, {})
    rows.append({"metric": name, "n_demo": len(y), "demo_auroc": auc, "demo_ci": f"[{lo:.2f}, {hi:.2f}]",
                 "full_auroc": f.get("auroc", np.nan), "full_ci": f"[{f['ci'][0]:.2f}, {f['ci'][1]:.2f}]" if f.get("ci") else "—"})
res = pd.DataFrame(rows)
print(f"R_AB demo rows: {len(rab)} ({int(rab.y.sum())} ERROR / {int((rab.y == 0).sum())} CORRECT); full run: n=2686 (1822 / 864)\n")
print(res.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

print("\nPre-registered prediction verdicts (full run):", {k: v for k, v in data["p_tests"].items() if k != "criteria"})
d = data["analysis"]["criteria"]["judge_local_qwen8b_disg"]["delta_R_AB"]
print(f"PEER+TEXT − local judge (full run, R_AB): Δ={d['delta']:+.3f} CI [{d['ci'][0]:.3f}, {d['ci'][1]:.3f}]")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [2.2, 1]})
xx = np.arange(len(res)); w = 0.38
ax[0].bar(xx - w / 2, res.demo_auroc, w, label=f"demo subset (n≈{len(rab)})", color="#8fb3d9")
ax[0].bar(xx + w / 2, res.full_auroc, w, label="full held-out E (n=2686)", color="#1f4e79")
ax[0].axhline(0.5, color="grey", ls="--", lw=1)
ax[0].set_xticks(xx); ax[0].set_xticklabels(res.metric, rotation=30, ha="right")
ax[0].set_ylim(0.4, 1.0); ax[0].set_ylabel("AUROC (ERROR vs CORRECT)"); ax[0].set_title("Metric AUROC, R_AB regime"); ax[0].legend()
for lbl, yv, col in [("CORRECT", 0, "#2e8b57"), ("ERROR", 1, "#c0392b")]:
    v = rab[rab.y == yv].p_peer_text.values
    ax[1].hist(v, bins=12, range=(0, 1), alpha=0.6, label=f"{lbl} (n={len(v)})", color=col)
ax[1].axvline(frozen["fusion"]["threshold"], color="k", ls=":", label="frozen flag threshold")
ax[1].set_xlabel("p_peer_text (higher = more likely error)"); ax[1].set_title("PEER+TEXT score by gold-derived label"); ax[1].legend()
plt.tight_layout(); plt.show()